In [ ]:
import fal_client
import os

os.environ["FAL_KEY"] = "d60c8b26-b223-4159-8bd0-7230944c7ca2:a373617798ccf30f349819f858088a12"

import requests
import json

# Make the initial request
response = requests.post(
    url="https://queue.fal.run/fal-ai/nano-banana",
    headers={
        "Authorization": f"Key {os.environ['FAL_KEY']}",
        "Content-Type": "application/json"
    },
    json={
        "prompt": "A wooden chess piece of a knight with gray background",
        "num_images": 1,
        "aspect_ratio": "1:1",
        "output_format": "png"
    }
)

# Parse the response and extract request_id
response_data = response.json()
request_id = response_data.get("request_id")

print(f"Request ID: {request_id}")
status_url = response_data.get("status_url")
# get it
status_response = requests.get(
    url=status_url
)
status_response.json()["status"] == "COMPLETED"
status_response.json()["response_url"]
responses_url_raw = requests.get(
    url=status_response.json()["response_url"]
)
urls = [d["url"] for d in responses_url_raw.json()["images"]]
images = [requests.get(url).content for url in urls]

Request ID: 74216254-33a2-4037-b0b9-862febdc262a


In [ ]:
def generate_images(model_name, prompt, num_images=1):
    assert model_name in ["fal-ai/nano-banana"], "Unsupported model_name"
    assert 1 <= num_images <= 4, "num_images must be between 1 and 4"
    response = requests.post(
        url=f"https://queue.fal.run/{model_name}",
        headers={
            "Authorization": f"Key {os.environ['FAL_KEY']}",
            "Content-Type": "application/json"
        },
        json={
            "prompt": prompt,
            "num_images": num_images,
            "aspect_ratio": "1:1",
            "output_format": "png"
        }
    )
    response_data = response.json()
    request_id = response_data.get("request_id")
    status_url = response_data.get("status_url")
    return request_id, status_url

def get_status(status_url):
    status_response = requests.get(
        url=status_url
    )
    return status_response.json()["status"]

def get_images(status_url):
    responses_url_raw = requests.get(
        url=status_url
    ).json()["response_url"]
    response_json = requests.get(
        url=responses_url_raw
    ).json()["images"]
    return [requests.get(d["url"]).content for d in response_json]


In [65]:
import time
# request_id, status_url = generate_images("fal-ai/nano-banana", "A wooden chess king piece on gray background", num_images=2)
while True:
    status = get_status(status_url)
    print(f"Status: {status}")
    print("="*80)
    if status == "COMPLETED":
        break
    time.sleep(2)

images = get_images(status_url)

Status: COMPLETED


In [66]:
images[0]

b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x04\x00\x00\x00\x04\x00\x08\x02\x00\x00\x00\xf0\x7f\xbc\xd4\x00\x00\x00\x89zTXtRaw profile type iptc\x00\x00\x08\x99M\x8c1\x0e\x021\x0c\x04\xfb\xbc\xe2\x9e\x908\xeb\xb5]S\xd1Q\xf0\x81\xbb\\"!!\x81\xf8\x7fA\xa0\xe0\x98iV[L:_\xae\xa7\xe5\xf9z\x8c\xdb\xbd\xa7\xe5\x0b\x91\xaaC\x10\xd83\xa6?\x8a\x97\x96\x85}\xae\x8a\x9d\x85\xa0J6c\x18\xe8\x92a\x1cSc\xfb\xfc(\x88\xa3#\xfa\xd7\xc9\xcd\xb7\xeaA\xdd\x00\xad\xb6F\xf1\xd5\xbai\x847\xefu0\xbd\x01C\xeb"\xa2\xee\xfdZc\x00\x00\x02\x87iTXtXML:com.adobe.xmp\x00\x00\x00\x00\x00<?xpacket begin="\xef\xbb\xbf" id="W5M0MpCehiHzreSzNTczkc9d"?> <x:xmpmeta xmlns:x="adobe:ns:meta/" x:xmptk="XMP Core 5.5.0"> <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"> <rdf:Description rdf:about="" xmlns:Iptc4xmpExt="http://iptc.org/std/Iptc4xmpExt/2008-02-29/" xmlns:photoshop="http://ns.adobe.com/photoshop/1.0/" Iptc4xmpExt:DigitalSourceFileType="http://cv.iptc.org/newscodes/digitalsourcetype/trainedAlgorithmicM

In [38]:
def get_status(model_name, request_id):
    status_url = f"https://queue.fal.run/{model_name}/status/{request_id}"
    status_response = requests.get(
        url=status_url,
        headers={
            "Authorization": f"Key {os.environ['FAL_KEY']}",
            "Content-Type": "application/json"
        }
    )
    return status_response


get_status("fal-ai/nano-banana", request_id)

<Response [405]>

In [39]:
request_id

'170d982f-bc54-4048-90b9-6695f599ab26'

In [32]:
# printa le immagini
from PIL import Image
from io import BytesIO
for i, img_data in enumerate(images):
    img = Image.open(BytesIO(img_data))
    img.show(title=f"Generated Image {i+1}")

In [14]:
response_data

{'status': 'IN_QUEUE',
 'request_id': '74216254-33a2-4037-b0b9-862febdc262a',
 'response_url': 'https://queue.fal.run/fal-ai/nano-banana/requests/74216254-33a2-4037-b0b9-862febdc262a',
 'status_url': 'https://queue.fal.run/fal-ai/nano-banana/requests/74216254-33a2-4037-b0b9-862febdc262a/status',
 'cancel_url': 'https://queue.fal.run/fal-ai/nano-banana/requests/74216254-33a2-4037-b0b9-862febdc262a/cancel',
 'logs': None,
 'metrics': {},
 'queue_position': 1}

FalClientHTTPError: Cannot access application 'fal-ai/nano-banana'. Authentication is required to access this application.